In [4]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

CSV = Path("processminer-sheet-break-rare-event-dataset.csv")   
OUT = Path("figures_large"); OUT.mkdir(exist_ok=True)

WIDTH = 3.5          # printed width in inches (3.5 = one IEEE column)

N_ANALYSIS = 18405
N_TRAIN = int(0.60 * N_ANALYSIS)

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 8.5,
    "axes.titlesize": 9,
    "axes.labelsize": 8.5,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 7.5,
    "figure.titlesize": 9.5,
    "axes.linewidth": 0.6,
    "lines.linewidth": 0.9,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.pad": 2, "ytick.major.pad": 2,
    "axes.labelpad": 2.5, "axes.titlepad": 4,
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "savefig.dpi": 600,
})


def save(fig, name):
    fig.savefig(OUT / f"{name}.pdf", bbox_inches="tight", pad_inches=0.02)
    fig.savefig(OUT / f"{name}.png", bbox_inches="tight", pad_inches=0.02)
    plt.close(fig)


df = pd.read_csv(CSV, encoding="utf-8-sig")
df["DateTime"] = pd.to_datetime(df["DateTime"], format="mixed")

ev = df.index[df["DateTime"] == pd.Timestamp("1999-05-04 17:10")][0]
win = df.iloc[ev - 120: ev + 120]
cols = ["MachSpd", 'TotHead"', "FanPmpSpd", "SodAlumFlo"]

fig, axes = plt.subplots(len(cols), 1, figsize=(WIDTH, 0.95 * WIDTH + 0.9), sharex=True)
for ax, c in zip(axes, cols):
    ax.plot(win["DateTime"], win[c], color="C0")
    ax.axvline(df.loc[ev, "DateTime"], color="C0", ls="--", lw=0.9)
    ax.set_ylabel(c)
    ax.grid(axis="y", ls="--", lw=0.4, alpha=0.35)
loc = mdates.HourLocator(byhour=range(0, 24, 3))
axes[-1].xaxis.set_major_locator(loc)
axes[-1].xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))
axes[-1].set_xlabel("Time")
fig.align_ylabels(axes)
fig.tight_layout(h_pad=0.4)
save(fig, "raw")

train = df.iloc[:N_TRAIN].drop(columns=["DateTime", "SheetBreak"])
iqr = (train.quantile(0.75) - train.quantile(0.25)).sort_values(ascending=False)
top = iqr.index[:8].tolist()

fig, ax = plt.subplots(figsize=(WIDTH, 0.72 * WIDTH + 0.5))
bp = ax.boxplot([train[c].values for c in top], patch_artist=True, widths=0.55,
                flierprops=dict(marker="o", markersize=2.5, markeredgewidth=0.6, alpha=0.6),
                medianprops=dict(color="C1", lw=1.0),
                whiskerprops=dict(lw=0.7), capprops=dict(lw=0.7), boxprops=dict(lw=0.7))
for b in bp["boxes"]:
    b.set_facecolor("C0"); b.set_hatch("//"); b.set_edgecolor("black")
matplotlib.rcParams["hatch.linewidth"] = 0.6
ax.axhline(0, color="0.35", ls="--", lw=0.8)
ax.set_xticks(range(1, len(top) + 1))
ax.set_xticklabels(top, rotation=40, ha="right", rotation_mode="anchor")
ax.set_ylabel("Observed value")
ax.grid(axis="y", lw=0.4, alpha=0.3)
fig.tight_layout()
save(fig, "boxplot")

H = [10, 20, 30, 50]
ev_test = df["SheetBreak"].to_numpy()[int(0.80 * N_ANALYSIS):N_ANALYSIS]

def horizon_labels(e, h):
    win = np.lib.stride_tricks.sliding_window_view(e[1:], h)   # rows i+1..i+h
    return win.max(axis=1)                                       # length len(e)-h

PREV = [100 * horizon_labels(ev_test, h).mean() for h in H]

fig, ax = plt.subplots(figsize=(WIDTH, 0.74 * WIDTH))
ax.plot(H, PREV, marker="o", ms=5, color="C0", lw=1.2)
ax.set_xlabel("Forecast horizon")
ax.set_ylabel("Positive precursor prevalence (%)")
#ax.set_title("Horizon-dependent target prevalence")
ax.set_xticks([10, 20, 30, 40, 50])
ax.grid(lw=0.4, alpha=0.35)
fig.tight_layout()
save(fig, "fig_horizon_target_prevalence_test")


t = df["DateTime"].iloc[:N_ANALYSIS]
dt = t.diff().dt.total_seconds().div(60)
nominal = float(dt.mode().iloc[0])
gaps = dt[dt > nominal].to_numpy()
idx = np.arange(len(gaps))

fig, (a1, a2) = plt.subplots(2, 1, figsize=(WIDTH, 0.80 * WIDTH + 0.4), sharex=True,
                             gridspec_kw=dict(height_ratios=[2.3, 1], hspace=0.30))
for ax in (a1, a2):
    ax.bar(idx, gaps, color="C1", edgecolor="black", lw=0.35, width=0.8)
    ax.grid(axis="y", lw=0.4, alpha=0.3)
a1.set_ylabel("Gap duration (min)")
a1.set_ylim(0, 1.32 * gaps.max())   
a1.set_title("Observed temporal discontinuities")
a1.text(0.015, 0.97, "(a)", transform=a1.transAxes, fontweight="bold", va="top")
a1.text(0.985, 0.97, f"Nominal sampling interval = {nominal:.1f} min\n(Shown in zoomed panel)",
        transform=a1.transAxes, ha="right", va="top", fontsize=7.5,
        bbox=dict(boxstyle="square,pad=0.3", fc="white", ec="0.6", lw=0.6))
a2.set_ylim(0, 10)
a2.axhline(nominal, color="#1f5fd1", ls="--", lw=1.0, label=f"Nominal sampling interval ({nominal:.1f} min)")
a2.set_title("Zoomed view: 0–10 min", pad=3)
a2.set_ylabel("Gap duration (min)")
a2.set_xlabel("Temporal-gap index")
a2.text(0.015, 0.06, "(b)", transform=a2.transAxes, fontweight="bold", va="bottom",
        bbox=dict(fc="white", ec="none", pad=0.5))
a2.legend(loc="upper right", fontsize=7, framealpha=1, borderpad=0.3, handlelength=1.6)
fig.align_ylabels((a1, a2))
fig.subplots_adjust(left=0.17, right=0.98, top=0.93, bottom=0.13)
save(fig, "fig_temporal_gap_duration")

print("saved to", OUT.resolve())

saved to /home/semon/Extreme Event/dynamics-physics/figures_large
